In [1]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_recall_fscore_support, confusion_matrix
import lightgbm as lgb
import numpy as np
import pandas as pd

In [2]:
def random_forest_train(X_train, y_train):

    # 5. Train model (NO random_state needed for reproducibility in this context)
    model = RandomForestClassifier(n_estimators=1000)
    model.fit(X_train, y_train)

    return model

In [3]:
def random_forest_test(model, X_train, y_train, X_test, y_test):
    # 6. Evaluate
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    print(f"Train accuracy: {train_score:.3f}")
    print(f"Test accuracy: {test_score:.3f}")

In [ ]:
def lightgbm_train(X_train, y_train, X_test, y_test):

    train_data = lgb.Dataset(X_train, label=y_train)
    test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

    # Determine objective automatically
    num_classes = len(np.unique(y_train))
    if num_classes == 2:
        objective = "binary"
        metric = "binary_logloss"
        num_class = None
    else:
        objective = "multiclass"
        metric = "multi_logloss"
        num_class = num_classes

    # Define parameters
    params = {
        'objective': objective,
        'boosting_type': 'gbdt',
        'metric': metric,
        'learning_rate': 0.02, # Slower learning rate often helps generalization
        'num_leaves': 31,      # Reduced from 63 to prevent overfitting
        'max_depth': 7,        # Limit depth
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_data_in_leaf': 50, # Increased to reduce noise sensitivity
        'lambda_l1': 0.5,       # Stronger regularization
        'lambda_l2': 0.5,
        'scale_pos_weight': (np.sum(y_train == 0) / np.sum(y_train == 1)) * 1.5,
        # 'is_unbalance': True,   # CRITICAL: Tells model to weight the minority class (Buy) higher
        'verbose': -1,
        'n_jobs': -1,
        'seed': 42
    }
    if num_class is not None:
        params['num_class'] = num_class

    # Train model (new callback format)
    callbacks = [
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100)
    ]

    model = lgb.train(
        params=params,
        train_set=train_data,
        num_boost_round=2000,
        valid_sets=[train_data, test_data],
        valid_names=['train', 'test'],
        callbacks=callbacks
    )

    return model

In [ ]:
def comprehensive_evaluation(model, X_test, y_test, model_name=""):

    y_pred = None
    
    if model_name == 'RF':
        y_pred = model.predict(X_test)

    if model_name == 'LGB':

        y_proba = model.predict(X_test)

        if y_proba.ndim > 1 and y_proba.shape[1] > 1:
            # If it's probability array (n_samples, n_classes)
            y_pred = (y_proba[:, 1] > 0.55).astype(int)
        else:
            # If it's already probabilities for class44 1
            y_pred = (y_proba > 0.55).astype(int)
            
    

    print(f"\n{'='*50}")
    print(f"COMPREHENSIVE EVALUATION: {model_name}")
    print(f"{'='*50}")
    
    # Basic metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
    
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-Score:  {f1:.3f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"True Negatives:  {cm[0,0]} | False Positives: {cm[0,1]}")
    print(f"False Negatives: {cm[1,0]} | True Positives:  {cm[1,1]}")
    
    # Class distribution
    print(f"\nClass Distribution:")
    print(f"Class 0 (SELL): {sum(y_test == 0)} samples")
    print(f"Class 1 (BUY):  {sum(y_test == 1)} samples")
    
    return accuracy, precision, recall, f1

